In [1]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
import warnings
import matplotlib.pyplot as plt
import sys 
import os 
import pandas as pd
import anndata as ad
import scanpy as sc
import numpy as np
import seaborn as sns
from scipy.stats import linregress
import matplotlib.patches as mpatches
from scipy.stats import pearsonr, spearmanr
plt.rcParams["figure.figsize"]=4,4

warnings.filterwarnings("ignore")

In [ ]:
import requests
import sys
import os
import pandas as pd
import numpy as np
from pathlib import Path
import json
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

palette_trend = {'Inconsistent': 'gray', 'Increase in aging': '#E52B50', 'Decrease in aging': '#B0BF1A'}
def create_nodes_edges(net) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Create nodes and edges from the input network DataFrame."""

    edges = net
    gene_names = np.concatenate([net['source'].unique(), net['target'].unique()])
    nodes = pd.DataFrame({'node': gene_names})

    # Compute out-degree and in-degree
    out_deg = net.groupby("source").size().reset_index(name="out_degree").rename(columns={"source": "node"})
    in_deg = net.groupby("target").size().reset_index(name="in_degree").rename(columns={"target": "node"})

    # Merge degrees
    centrality = pd.merge(out_deg, in_deg, on="node", how="outer").fillna(0)
    centrality["centrality"] = centrality["in_degree"] + centrality["out_degree"]

    # Trend from both source and target
    meta_source = net[['source', 'trend_source']].drop_duplicates().rename(columns={'source': 'node', 'trend_source': 'trend'})
    meta_target = net[['target', 'trend_target']].drop_duplicates().rename(columns={'target': 'node', 'trend_target': 'trend'})

    # Combine trends: source has priority
    meta_combined = pd.concat([meta_target, meta_source]).drop_duplicates(subset='node', keep='last')

    # Merge info into nodes
    nodes = nodes.merge(centrality[['node', 'centrality']], on='node', how='left')
    nodes = nodes.merge(meta_combined, on='node', how='left')

    # Fill missing values
    nodes['centrality'] = nodes['centrality'].fillna(1)
    nodes['trend'] = nodes['trend'].fillna('Unknown')

    # Assign color using trend
    nodes['color'] = nodes['trend'].map(palette_trend).fillna('#FFFFFF')  # fallback to white

    return nodes, edges

class PlotCytoscape:
    palette_regulation = {'Positive': '#009E73', 'Negative': 'lightcoral'}
    

    def __init__(self, nodes, edges, node_size_attribute=None, edge_color_attribute=None, edge_size_attribute=None):
        assert (len(nodes) > 0)
        assert (len(edges) > 0)
        # - scale node size
        if node_size_attribute is not None:
            scale, min_value = 20, 20
            nodes["size"] = self.normalize(nodes[node_size_attribute], scale=scale, min_value=min_value)
        # - scale edge width
        if edge_size_attribute is not None:
            scale, min_value = 3, 1
            # edges["width"] = self.normalize(edges[edge_size_attribute], scale=scale,
            #                                              min_value=min_value)
            edges["width"] = 1
        # - assign color to edges
        # if edge_color_attribute is not None:
        #     edges['edge_color'] = self.assign_edge_colors(edges[edge_color_attribute])
        edges['edge_color'] = 'black'
        # - assign arrow shape and size
        edges["arrow_shape"] = self.assign_arrow_shapes(edges["weight"])
        edges["arrow_size"] = self.assign_arrow_sizes(edges["weight"])


        self.base = 'http://localhost:' + str(1234) + '/v1/'
        self.ID = None
        self.nodes = self.extract_dict(nodes.rename(columns={'node': 'id'}))
        self.edges = self.extract_dict(edges)

    # @classmethod
    # def assign_edge_colors(cls, values):
    #     colors = []
    #     for val in values:
    #         if val >= 0:
    #             colors.append(cls.palette_regulation['Positive'])
    #         else:
    #             colors.append(cls.palette_regulation['Negative'])
    #     return colors
    @classmethod
    def assign_arrow_shapes(cls, values):
        arrows = []
        for val in values:
            if val >= 0:
                arrows.append('DELTA_SHORT_1')       # or 'DELTA'
            else:
                arrows.append('T')           # 'T' is the blocking head in Cytoscape
        return arrows
    def assign_arrow_sizes(self, values):
        return [4 if v < 0 else 8 for v in values]
    @staticmethod
    def normalize(values, scale=1, min_value=0):
        min_val = min(values)
        max_val = max(values)
        if min_val == max_val:
            return values
        normalized_values = [min_value+ (val - min_val) / (max_val - min_val)*scale for val in values]
        return normalized_values
    @staticmethod
    def extract_dict(df) -> List[Dict]:
        """
        Puts the df into a list of dict where it presents each row of the df
        """
        nodes_dict = df.to_dict('records')
        list_of_data = []
        for node in nodes_dict:
            data = {'data': {}}
            for col in df.columns:
                data['data'][col] = node[col]
            list_of_data.append(data)
        return list_of_data
    def _delete_network(self, name):
        """
            If the name exist, delete the network before creating it
        """
        response = requests.get(self.base + 'networks.names')
        networks = response.json()
        network_suid = None
        for network in networks:
            if network['name'] == name:
                network_suid = network['SUID']
                break

        # Delete the network if it exists
        if network_suid is not None:
            response = requests.delete(self.base + f'networks/{network_suid}')
            if response.status_code == 200:
                print(f"Network '{name}' deleted successfully")
            else:
                print(f"Failed to delete network '{name}'")
                print(response.status_code)
                print(response.text)
        else:
            print(f"No network found with name '{name}'")
    def create_network(self, name='mynetwork'):
        self._delete_network(name)
        HEADERS = {'Content-Type': 'application/json'}
        elements = {"nodes": self.nodes, "edges": self.edges}
        print(elements['edges'])
        network = {
            'data': {
                'name': name
            },
            'elements': elements,
        }

        response = requests.post(self.base + 'networks?collection=My%20Collection', data=json.dumps(network),
                                 headers=HEADERS)
        print(response.text)
        res_dict = response.json()
        new_suid = res_dict['networkSUID']
        self.ID = new_suid
        print(f"Network '{name}' created successfully")
    @staticmethod
    def check(response, code, name):
        if response.status_code == code:
            print(f"{name} successful")
        else:
            print(f"Failed to {name}")
            print(response.status_code)
            print(response.text)
    @staticmethod
    def passthrough_map(style_data, attribute, attribute_type, visual_property):
        """
            Used to map the given attribute to the visual property. The column attribute should be present.
        """
        mapping = {
            "mappingType": "passthrough",
            "mappingColumn": attribute,
            "mappingColumnType": attribute_type,
            "visualProperty": visual_property
        }
        style_data['mappings'].append(mapping)
    @staticmethod
    def single_value_map(style_data, visual_property, value):
        """
            Used to set a single value to the given visual property
        """
        mapping = {
            "visualProperty": visual_property,
            "value" : value
        }
        style_data['defaults'].append(mapping)

    def make_changes(self, style_name="CustomStyle"):
        # - apply layout
        response = requests.get(self.base + f'apply/layouts/force-directed/{self.ID}')
        self.check(response, 200, 'directed force')
        # - get the Directed style
        response = requests.get(self.base + 'styles/Directed')
        self.check(response, 200, 'directed')
        directed_style = response.json()
        # - Create a new style
        custom_style = directed_style
        custom_style['title'] = style_name
        # - make changes
        self.passthrough_map(custom_style, "color", 'String', 'NODE_FILL_COLOR')
        self.passthrough_map(custom_style, "size", 'Float', 'NODE_SIZE')
        self.passthrough_map(custom_style, "width", 'Float', 'EDGE_WIDTH')
        self.passthrough_map(custom_style, "edge_color", 'String', 'EDGE_STROKE_UNSELECTED_PAINT')
        self.passthrough_map(custom_style, "edge_color", 'String', 'EDGE_TARGET_ARROW_UNSELECTED_PAINT')
        self.passthrough_map(custom_style, "edge_color", 'String', 'EDGE_TARGET_ARROW_UNSELECTED_PAINT')
        self.single_value_map(custom_style, "NODE_LABEL_COLOR", "#000000")
        self.single_value_map(custom_style, "NODE_BORDER_WIDTH", 1)
        self.single_value_map(custom_style, "NODE_LABEL_FONT_SIZE", 8)
        self.passthrough_map(custom_style, "arrow_shape", "Float", "EDGE_TARGET_ARROW_SHAPE")
        self.passthrough_map(custom_style, "arrow_size", "Float", "EDGE_TARGET_ARROW_SIZE")
        # - delete the previous style with the same name
        response = requests.delete(self.base + f'styles/{style_name}')
        self.check(response, 200, 'delete styles')
        # - add style
        response = requests.post(self.base + 'styles', json=custom_style)
        self.check(response, 201, 'custom style added')
        # - apply style
        response = requests.get(self.base + f'apply/styles/{style_name}/{self.ID}')
        self.check(response, 200, 'apply style')


In [137]:
def get_tf_family():
    tf_family = pd.read_csv('../output/TableS1.csv')
    tf_family.rename(columns={'Unnamed: 1': 'TF', 'Unnamed: 2': 'family'}, inplace=True)
    tf_family = tf_family[['TF', 'family']].drop_duplicates().set_index('TF')
    return tf_family


In [147]:
!vtools use HGNC --linked_by refGene.name2

/bin/bash: vtools: command not found


python(53825) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [141]:
net = pd.read_csv('../output/collapsed_net_CD8T.csv')

# - add additional info to the net
net['target_genes'] = net['target'].apply(lambda x: x.split('/'))
net['source_genes'] = net['source'].apply(lambda x: x.split('/'))
net['target_n'] = net['target_genes'].apply(lambda x: len(x))
net['source_n'] = net['source_genes'].apply(lambda x: len(x))

# - add tf family
tf_family = get_tf_family()

net['tf_family'] = net['source'].apply(lambda x: ', '.join(tf_family.loc[x.split('/')]['family'].unique()))

# - replace the source with tf family if the source has more than 20 names
net['source_o'] = net['source']
net['source'] = np.where(
    net['source_n'] > 20,
    net['tf_family'],
    net['source_o']
)
net.head()

,source,target,weight,trend_source,trend_target,sign,target_genes,source_genes,target_n,source_n,tf_family,source_o
0,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",A1BG/ABLIM1/ACTN1/AIF1/AK5/ALKBH7/ANP32B/APBA2...,-1.000000,Increase in aging,Decrease in aging,-1.0,"[A1BG, ABLIM1, ACTN1, AIF1, AK5, ALKBH7, ANP32...","[AKNA, ARID5B, ASCL2, BATF, BHLHE40, EOMES, FO...",170,32,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",AKNA/ARID5B/ASCL2/BATF/BHLHE40/EOMES/FOSL2/GFI...
1,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",ABHD17A/ABI3/ACTB/ACTN4/ADAM8/ADRB2/ALOX5AP/AN...,0.998217,Increase in aging,Increase in aging,1.0,"[ABHD17A, ABI3, ACTB, ACTN4, ADAM8, ADRB2, ALO...","[AKNA, ARID5B, ASCL2, BATF, BHLHE40, EOMES, FO...",332,32,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",AKNA/ARID5B/ASCL2/BATF/BHLHE40/EOMES/FOSL2/GFI...
2,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",ADPRM/AQP3/ATF7IP2/ATM/ATP6V0E2/ATP6V1G1/C1QBP...,-1.000000,Increase in aging,Decrease in aging,-1.0,"[ADPRM, AQP3, ATF7IP2, ATM, ATP6V0E2, ATP6V1G1...","[AKNA, ARID5B, ASCL2, BATF, BHLHE40, EOMES, FO...",53,32,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",AKNA/ARID5B/ASCL2/BATF/BHLHE40/EOMES/FOSL2/GFI...
3,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",ATP1B3/GSTK1/TUBA4A/VMP1,1.000000,Increase in aging,Increase in aging,1.0,"[ATP1B3, GSTK1, TUBA4A, VMP1]","[AKNA, ARID5B, ASCL2, BATF, BHLHE40, EOMES, FO...",4,32,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",AKNA/ARID5B/ASCL2/BATF/BHLHE40/EOMES/FOSL2/GFI...
4,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",CCNH/RGS1/ZFP36L1,1.000000,Increase in aging,Mixed,1.0,"[CCNH, RGS1, ZFP36L1]","[AKNA, ARID5B, ASCL2, BATF, BHLHE40, EOMES, FO...",3,32,"AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ...",AKNA/ARID5B/ASCL2/BATF/BHLHE40/EOMES/FOSL2/GFI...


In [156]:
# import requests

# url = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/locus_groups/protein-coding_gene.txt"
# response = requests.get(url)

# with open("protein-coding_gene.csv", "wb") as f:
#     f.write(response.content)

In [176]:
import pandas as pd

# Load HGNC gene family data
hgnc_df = pd.read_csv('protein-coding_gene.csv', sep='\t')  # Replace with your actual file path


gene_to_family = hgnc_df.set_index('symbol')['gene_group'].to_dict()

# Function to map a list of genes to their families
def map_to_families(gene_list):
    families = [gene_to_family.get(gene, 'Unknown') for gene in gene_list]
    print(families)
    families = ', '.join(families)  # Join families into a single string
    return list(set(families))  # Remove duplicates

# Apply to your net DataFrame
net['target_gene_families'] = net['target_genes'].apply(map_to_families)
net

['Immunoglobulin like domain containing', 'LIM domain containing', 'Actinins', 'EF-hand domain containing', 'Adenylate kinases', 'Alkylation repair homologs', 'ANP32 acidic nuclear phosphoproteins', 'PDZ domain containing', 'Armadillo like helical domain containing', 'Basic leucine zipper proteins|BTB domain containing', 'Zinc fingers C2H2-type|BAF complex subunits', nan, 'Calmodulin dependent protein kinases', 'CD molecules|C-C motif chemokine receptors', 'Blood group antigens|CD molecules|Sushi domain containing|Complement system regulators and receptors', 'CD molecules|V-set domain containing', 'Charged multivesicular body proteins|ESCRT-III associated factors', 'Unknown', 'C-type lectin domain containing', nan, 'Mitochondrial complex IV: cytochrome c oxidase subunits', 'Mitochondrial complex IV: cytochrome c oxidase subunits|MicroRNA protein coding host genes', 'EF-hand domain containing|Diacylglycerol kinases', nan, nan, 'Small nucleolar RNA protein coding host genes', 'MicroRNA p

TypeError: sequence item 11: expected str instance, float found

In [ ]:
# - formatt the net for cytoscape
cell_type = 'CD8T'

nodes, edges = create_nodes_edges(net)

# - send them to cytoscape
obj = PlotCytoscape(nodes, edges, node_size_attribute='centrality', 
                    edge_color_attribute='weight',
                    edge_size_attribute='weight')

obj.create_network(name=f'targets_{cell_type}')
obj.make_changes(style_name=f'style_targets_{cell_type}')

Network 'targets_CD8T' deleted successfully
[{'data': {'source': 'AT hook, ARID/BRIGHT, bHLH, bZIP, T-box, C2H2 ZF, IRF, Myb/SANT, Rel, Nuclear receptor, Runt, SAND, STAT, E2F, C2H2 ZF; Homeodomain', 'target': 'A1BG/ABLIM1/ACTN1/AIF1/AK5/ALKBH7/ANP32B/APBA2/ARMH1/BACH2/BCL11B/C12orf57/CAMK4/CCR7/CD55/CD8B/CHMP7/CHRM3-AS2/CLEC11A/CNN2/COX4I1/COX7C/DGKA/DNPH1/EEF1A1/EEF1B2/EEF1G/EEF2/EIF3E/EIF3H/EPHA1-AS1/EPHX2/FBL/FCMR/FHIT/FLT3LG/FOXP1/GRAMD1A/GYPC/HNRNPA1/HSP90AB1/HSPB1/ID3/IL6R/IL6ST/IL7R/IMPDH2/ITGA6/LDLRAP1/LEF1/LGALS3BP/LINC01550/LINC02273/LINC02446/LRRN3/MAL/MAML2/MDS2/MYC/NACA/NAP1L1/NCF1/NDFIP1/NELL2/NOG/NOSIP/NPM1/NT5E/NUCB2/OXNAD1/PABPC1/PASK/PCED1B/PDE3B/PLAG1/PRKCA/PRKCQ-AS1/RAB3GAP1/RACK1/RCAN3/REG4/RETREG1/RGS10/RPL10A/RPL11/RPL12/RPL13/RPL13A/RPL14/RPL17/RPL18/RPL18A/RPL19/RPL22/RPL23/RPL23A/RPL24/RPL26/RPL27/RPL27A/RPL28/RPL29/RPL30/RPL32/RPL35/RPL35A/RPL37/RPL37A/RPL38/RPL4/RPL5/RPL6/RPL7A/RPL8/RPL9/RPLP0/RPLP1/RPLP2/RPS11/RPS12/RPS13/RPS15A/RPS16/RPS17/RPS21/RPS23/RPS

In [143]:
n_genes_t = 20
targets = net[net['target_n'] > n_genes_t]['target'].unique()
genes_list = np.concatenate([targets])


In [85]:
n_keep_terms = 5

map_gs_terms = {}
for gs in genes_list[2:3]:
    genes = gs.split('/')
    # print(genes)
    # - run ora
    import gseapy as gp
    rr = gp.enrichr(gene_list=list(genes),
                                    gene_sets=['GO_Molecular_Function_2021'], #, 'KEGG_2021_Human' ,'MSigDB_Hallmark_2020', 'GO_Molecular_Function_2021'
                                    organism='human', 
                                    outdir=None, 
                                    cutoff=1
                                    # background=list(tf_all),
                                    )
    res2d = rr.res2d
    print(res2d)
    res2d_sig = res2d[res2d['Adjusted P-value']<0.05]
    if res2d_sig.shape[0]>n_keep_terms:
        res2d_sig['n_genes'] = res2d_sig['Overlap'].apply(lambda x: x.split('/')[0]).astype(int)
        res2d_sig = res2d_sig.sort_values(by='n_genes', ascending=False).head(n_keep_terms)
    terms = res2d_sig['Term'].apply(lambda x: x.split('(')[0]).values
    terms = [term.strip() for term in terms]
    terms = '/'.join(terms)
    map_gs_terms[gs] = terms
    

                      Gene_set  \
0   GO_Molecular_Function_2021   
1   GO_Molecular_Function_2021   
2   GO_Molecular_Function_2021   
3   GO_Molecular_Function_2021   
4   GO_Molecular_Function_2021   
..                         ...   
95  GO_Molecular_Function_2021   
96  GO_Molecular_Function_2021   
97  GO_Molecular_Function_2021   
98  GO_Molecular_Function_2021   
99  GO_Molecular_Function_2021   

                                                 Term Overlap   P-value  \
0                       ribosome binding (GO:0043022)    2/48  0.007189   
1               pyrophosphatase activity (GO:0016462)    2/54  0.009029   
2   histone demethylase activity (H3-K27 specific)...     1/5  0.013181   
3          RNA strand annealing activity (GO:0033592)     1/5  0.013181   
4      ADP-ribose diphosphatase activity (GO:0047631)     1/5  0.013181   
..                                                ...     ...       ...   
95                           DNA binding (GO:0003677)   2/811  0.6

In [88]:
res2d
np.asarray(genes)

array(['ADPRM', 'AQP3', 'ATF7IP2', 'ATM', 'ATP6V0E2', 'ATP6V1G1', 'C1QBP',
       'CCNI', 'CD28', 'CDCA7L', 'CPNE2', 'DPP4', 'DSEL', 'EIF4A2',
       'EIF4B', 'ESD', 'GIMAP1', 'GPR155', 'HAPLN3', 'HSBP1L1', 'HSPD1',
       'IL23A', 'ILF3-DT', 'JAML', 'KDM6B', 'KLF7', 'LBH', 'LINC01215',
       'LYRM4', 'MEST', 'MLXIP', 'MZF1-AS1', 'NAA16', 'NFKBIZ', 'NME2',
       'NSA2', 'PADI4', 'PAG1', 'PDK1', 'PHGDH', 'PLK3', 'PNISR', 'PPA1',
       'PRKAR1B', 'RIPOR2', 'RNF157', 'RUNX2', 'SNHG25', 'STMN3', 'THEM4',
       'TNFRSF25', 'ZNF101', 'ZSCAN18'], dtype='<U9')

In [84]:
list(map_gs_terms.values())

['']

family    C2H2 ZF
Name: KLF6, dtype: object